# Introducción a Patrones Arquitectónicos de Datos

## Objetivos de Aprendizaje

- Comprender qué es un **patrón arquitectónico** y por qué es útil
- Conocer la **arquitectura en capas** (Bronze/Silver/Gold)
- Diferenciar arquitecturas **monolíticas vs distribuidas**
- Introducción a **event-driven architecture**
- Entender el concepto de **reutilización de datos**

## 1. ¿Qué es un Patrón Arquitectónico?

Un **patrón arquitectónico** es una solución probada y reutilizable a problemas comunes en arquitectura de sistemas.

### ¿Por qué usar patrones?

✅ **Ventajas:**
- **Reutilización:** No reinventar la rueda
- **Comunicación:** Lenguaje común entre arquitectos
- **Mejores prácticas:** Soluciones probadas en producción
- **Mantenibilidad:** Código más estructurado y comprensible

❌ **Desventajas:**
- No existe un patrón único para todos los casos
- Puede agregar complejidad innecesaria si se aplica mal
- Requiere entender el contexto y trade-offs

### Ejemplos de patrones comunes

| Patrón | Descripción | Uso en Datos |
|--------|-------------|---------------|
| **Layered (Capas)** | Organizar en capas con responsabilidades separadas | Bronze → Silver → Gold |
| **Event-Driven** | Comunicación asíncrona mediante eventos | Streaming de datos, CDC |
| **Microservicios** | Componentes independientes y especializados | Data products, APIs de dominio |
| **Pub/Sub** | Productores publican, consumidores se suscriben | Kafka, Event Hub |
| **ETL/ELT** | Extract-Transform-Load / Extract-Load-Transform | Pipelines de datos |

## 2. Arquitectura en Capas: Bronze/Silver/Gold

El patrón **Medallion Architecture** organiza datos en 3 capas de calidad creciente.

### 🥉 Capa Bronze (Raw)

**Propósito:** Ingestar datos tal como vienen de las fuentes

**Características:**
- Datos **crudos** sin transformación
- Schema-on-read (flexible)
- Formatos: JSON, CSV, Parquet raw
- **Inmutable:** Se preserva el dato original
- Retención: largo plazo (años)

**Ejemplo:**
```
bronze/
  ├── ecommerce/
  │   ├── orders_raw/       # Pedidos sin procesar
  │   └── customers_raw/    # Clientes sin limpiar
  └── logs/
      └── app_events/       # Logs de aplicación
```

### 🥈 Capa Silver (Curated)

**Propósito:** Limpiar, validar y estructurar datos

**Características:**
- Datos **curados** (cleaned)
- Deduplicación, normalización
- Tipos de datos validados
- Enriquecimiento básico
- Retención: mediano plazo

**Transformaciones típicas:**
- Eliminar duplicados
- Convertir tipos (strings → dates)
- Estandarizar nombres de columnas
- Filtrar registros inválidos

**Ejemplo:**
```
silver/
  ├── ecommerce/
  │   ├── orders_cleaned/      # Sin duplicados, tipos correctos
  │   └── customers_validated/ # Emails válidos, formato estándar
```

### 🥇 Capa Gold (Trusted)

**Propósito:** Datos listos para consumo analítico y BI

**Características:**
- Datos **agregados** y **optimizados**
- Modelos dimensionales (star schema)
- Métricas de negocio pre-calculadas
- Alto rendimiento de lectura
- Retención: según políticas de negocio

**Ejemplo:**
```
gold/
  ├── bi/
  │   ├── fact_sales/         # Tabla de hechos de ventas
  │   ├── dim_customers/      # Dimensión clientes
  │   └── dim_products/       # Dimensión productos
  └── kpis/
      └── daily_revenue/      # KPI agregado diario
```

### Flujo de Datos

```
Fuentes → [Bronze] → Limpieza → [Silver] → Agregación → [Gold] → BI/ML
          (Raw)                  (Curated)              (Trusted)
```

In [ ]:
# Simulación de transformación Bronze → Silver → Gold
import pandas as pd
import numpy as np

# === BRONZE: Datos crudos ===
bronze_data = pd.DataFrame({
    'order_id': ['001', '002', '003', '003', '004', None],  # Duplicado y nulo
    'customer_email': ['juan@example.com', 'MARIA@EXAMPLE.COM', 'invalid-email', 
                       'pedro@example.com', 'pedro@example.com', 'ana@example.com'],
    'amount': ['100.50', '250', 'invalid', '150.75', '150.75', '300'],  # String con valores inválidos
    'order_date': ['2025-01-15', '2025-01-16', '2025-13-99', '2025-01-17', '2025-01-17', '2025-01-18']
})

print("=== BRONZE (Raw Data) ===")
print(bronze_data)
print(f"\nRegistros: {len(bronze_data)}")
print(f"Tipos de datos:\n{bronze_data.dtypes}")

In [ ]:
# === SILVER: Limpieza y validación ===

# 1. Eliminar registros con order_id nulo
silver_data = bronze_data.dropna(subset=['order_id']).copy()

# 2. Eliminar duplicados (por order_id)
silver_data = silver_data.drop_duplicates(subset=['order_id'])

# 3. Normalizar emails a minúsculas
silver_data['customer_email'] = silver_data['customer_email'].str.lower()

# 4. Validar formato de email
import re
email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
silver_data['email_valid'] = silver_data['customer_email'].apply(
    lambda x: bool(re.match(email_pattern, str(x)))
)
silver_data = silver_data[silver_data['email_valid']].drop(columns=['email_valid'])

# 5. Convertir amount a numérico (coerce invalids to NaN)
silver_data['amount'] = pd.to_numeric(silver_data['amount'], errors='coerce')
silver_data = silver_data.dropna(subset=['amount'])

# 6. Validar y convertir fechas
silver_data['order_date'] = pd.to_datetime(silver_data['order_date'], errors='coerce')
silver_data = silver_data.dropna(subset=['order_date'])

# Reset index
silver_data = silver_data.reset_index(drop=True)

print("=== SILVER (Curated Data) ===")
print(silver_data)
print(f"\n✅ Registros después de limpieza: {len(silver_data)} (de {len(bronze_data)} originales)")
print(f"📉 Registros eliminados: {len(bronze_data) - len(silver_data)}")
print(f"\nTipos de datos validados:\n{silver_data.dtypes}")

In [ ]:
# === GOLD: Agregación para BI ===

# Agregar por día
gold_daily_revenue = silver_data.groupby(
    silver_data['order_date'].dt.date
).agg({
    'order_id': 'count',
    'amount': 'sum',
    'customer_email': 'nunique'
}).rename(columns={
    'order_id': 'total_orders',
    'amount': 'total_revenue',
    'customer_email': 'unique_customers'
})

# Calcular ticket promedio
gold_daily_revenue['avg_order_value'] = (
    gold_daily_revenue['total_revenue'] / gold_daily_revenue['total_orders']
).round(2)

print("=== GOLD (Aggregated for BI) ===")
print(gold_daily_revenue)
print(f"\n📊 KPI: Revenue total = ${gold_daily_revenue['total_revenue'].sum():.2f}")
print(f"📊 KPI: Clientes únicos = {gold_daily_revenue['unique_customers'].sum()}")
print(f"📊 KPI: Ticket promedio = ${gold_daily_revenue['avg_order_value'].mean():.2f}")

## 3. Arquitecturas: Monolítica vs Distribuida

### 🏢 Arquitectura Monolítica

**Definición:** Toda la funcionalidad en una sola aplicación/base de datos.

**Características:**
- Código en un solo repositorio
- Base de datos única y centralizada
- Despliegue como una unidad

**✅ Ventajas:**
- Simple de desarrollar inicialmente
- Fácil de debuggear (todo en un lugar)
- Transacciones ACID simples
- Menor latencia interna (sin red)

**❌ Desventajas:**
- Difícil de escalar (todo o nada)
- Cambios requieren redespliegue completo
- Un error puede tumbar todo el sistema
- Equipos grandes tienen conflictos de código

**Ejemplo:**
```
┌─────────────────────────────────┐
│   Aplicación Monolítica         │
│  ┌──────────────────────────┐   │
│  │  ETL + Transformaciones  │   │
│  │  Reportes + BI           │   │
│  │  APIs + Consultas        │   │
│  └──────────────────────────┘   │
│            ↕                     │
│  ┌──────────────────────────┐   │
│  │  Base de Datos Única     │   │
│  └──────────────────────────┘   │
└─────────────────────────────────┘
```

### 🌐 Arquitectura Distribuida

**Definición:** Funcionalidad dividida en múltiples servicios/componentes independientes.

**Características:**
- Múltiples servicios especializados
- Cada servicio con su propia BD (opcional)
- Comunicación por red (APIs, eventos)
- Despliegue independiente

**✅ Ventajas:**
- Escala independiente por componente
- Equipos autónomos (ownership)
- Tecnologías especializadas por servicio
- Fallas aisladas (resiliencia)

**❌ Desventajas:**
- Mayor complejidad operacional
- Latencia de red entre servicios
- Transacciones distribuidas complejas
- Debugging más difícil

**Ejemplo:**
```
┌─────────────┐   ┌─────────────┐   ┌─────────────┐
│  Ingesta    │   │ Procesamiento│   │   Serving   │
│  Service    │──▶│   Service    │──▶│   Service   │
│             │   │              │   │             │
│  (Kafka)    │   │  (Spark)     │   │  (APIs)     │
└─────────────┘   └─────────────┘   └─────────────┘
      ↕                  ↕                  ↕
   [Raw DB]         [Processed]         [Gold DB]
```

### Comparación

| Aspecto | Monolítica | Distribuida |
|---------|------------|-------------|
| **Complejidad inicial** | Baja | Alta |
| **Escalabilidad** | Vertical | Horizontal |
| **Despliegue** | Todo junto | Independiente |
| **Resiliencia** | Fallo = todo cae | Fallo aislado |
| **Performance** | Baja latencia interna | Latencia de red |
| **Equipos** | 1-5 personas | Múltiples equipos |
| **Cuándo usar** | MVP, proyectos pequeños | Escala, equipos grandes |

## 4. Event-Driven Architecture (Arquitectura Orientada a Eventos)

### ¿Qué es?

Un patrón donde los componentes se comunican mediante **eventos** (mensajes asíncronos) en lugar de llamadas directas.

### Componentes Clave

1. **Event Producer (Productor):** Genera eventos cuando algo ocurre
2. **Event Bus/Broker:** Transporta eventos (Kafka, RabbitMQ, Event Hub)
3. **Event Consumer (Consumidor):** Procesa eventos de forma asíncrona

### Ejemplo: E-commerce

```
Order Service          Event Bus (Kafka)       Consumers
     │                       │                    │
     │  1. order.created     │                    │
     ├──────────────────────▶│                    │
     │                       │  2. consume        │
     │                       ├───────────────────▶│ Inventory Service
     │                       │                    │   (reduce stock)
     │                       ├───────────────────▶│ Email Service
     │                       │                    │   (send confirmation)
     │                       ├───────────────────▶│ Analytics Service
     │                       │                    │   (update metrics)
```

### Ventajas

✅ **Desacoplamiento:** Productores y consumidores no se conocen
✅ **Escalabilidad:** Agregar consumidores sin modificar productores
✅ **Asincronía:** No bloquea el flujo principal
✅ **Replay:** Reprocesar eventos históricos

### Desventajas

❌ **Eventual consistency:** Los datos no son inmediatamente consistentes
❌ **Debugging complejo:** Flujos distribuidos difíciles de trazar
❌ **Orden no garantizado:** (a menos que se use particionamiento por clave)

### Casos de Uso en Datos

- **CDC (Change Data Capture):** Capturar cambios de BD y propagarlos
- **Real-time analytics:** Procesar eventos de clickstream, IoT
- **Notificaciones:** Alertas basadas en eventos de datos
- **Data pipelines:** Orquestar transformaciones mediante eventos

## 5. Reutilización de Datos

### Principio: "Don't Repeat Yourself" (DRY)

En arquitectura de datos, esto significa:
- **No duplicar lógica de transformación** (centralizar en capa Silver)
- **No crear múltiples copias** de los mismos datos procesados
- **Definir datasets canónicos** que múltiples equipos consuman

### Patrón: Single Source of Truth (SSOT)

**Definición:** Cada entidad de negocio tiene una única representación autoritativa.

**Ejemplo:**
```
❌ Mal diseño:
- Marketing tiene su propia tabla de clientes
- Sales tiene otra tabla de clientes
- Support tiene otra más
→ Inconsistencias, duplicación de esfuerzo

✅ Buen diseño:
- Una tabla gold.dim_customers (SSOT)
- Todos los equipos consumen esta tabla
→ Consistencia, mantenimiento centralizado
```

### Estrategias de Reutilización

1. **Datasets compartidos:**
   - Catálogo de datos con descripciones claras
   - Versionado de esquemas
   - SLAs de disponibilidad

2. **APIs de datos:**
   - Exponer datos via REST/GraphQL
   - Permite control de acceso granular
   - Abstrae implementación interna

3. **Views y Tablas derivadas:**
   - Vistas materializadas para casos comunes
   - Reducir duplicación de queries complejas

4. **Data Contracts:**
   - Acuerdos formales entre productores y consumidores
   - Garantías de esquema, calidad, SLAs

### Beneficios

📈 **Eficiencia:** Menos código duplicado, menos mantenimiento
🎯 **Consistencia:** Una sola versión de la verdad
💰 **Costo:** Menos procesamiento redundante
⚡ **Velocidad:** Teams no reinventan soluciones existentes

## 6. Ejercicio Práctico

**Escenario:** Sistema de e-commerce con los siguientes requisitos:

1. **Ingesta:** Pedidos, clientes, productos desde BD transaccional
2. **Procesamiento:** Limpiar datos, calcular métricas
3. **Consumo:** Dashboard de ventas, recomendaciones ML

**Tareas:**

### A. Diseño en Capas (30%)
- Dibujar arquitectura Bronze/Silver/Gold
- Listar qué tablas/archivos van en cada capa
- Describir transformaciones entre capas

### B. Monolítica vs Distribuida (30%)
- Evaluar qué arquitectura usar para:
  - Startup con 2 data engineers
  - Empresa grande con 20+ data engineers
- Justificar decisión con 3 razones

### C. Event-Driven (20%)
- Identificar 3 eventos de negocio (ej: "order.created")
- Describir qué consumidores los procesarían
- ¿Qué ventajas aporta vs polling de BD cada minuto?

### D. Reutilización (20%)
- Identificar 2 datasets que deberían ser SSOT
- Proponer cómo evitar duplicación de lógica
- Listar 3 métricas que múltiples equipos consumirían

In [ ]:
# Plantilla de solución
solucion = '''
=== SOLUCIÓN EJERCICIO ===

A. DISEÑO EN CAPAS:

Bronze (Raw):
  - [listar tablas/archivos]

Silver (Curated):
  - [listar tablas transformadas]
  
Gold (Trusted):
  - [listar modelos dimensionales]

Transformaciones:
  Bronze → Silver: [describir]
  Silver → Gold: [describir]

B. MONOLÍTICA VS DISTRIBUIDA:

Startup (2 engineers):
  Arquitectura: [Monolítica / Distribuida]
  Razones:
    1. [razón 1]
    2. [razón 2]
    3. [razón 3]

Empresa grande (20+ engineers):
  Arquitectura: [Monolítica / Distribuida]
  Razones:
    1. [razón 1]
    2. [razón 2]
    3. [razón 3]

C. EVENT-DRIVEN:

Eventos de negocio:
  1. [evento 1] → Consumidores: [lista]
  2. [evento 2] → Consumidores: [lista]
  3. [evento 3] → Consumidores: [lista]

Ventajas vs polling:
  - [ventaja 1]
  - [ventaja 2]
  - [ventaja 3]

D. REUTILIZACIÓN:

Datasets SSOT:
  1. [dataset 1]: [descripción]
  2. [dataset 2]: [descripción]

Evitar duplicación:
  - [estrategia]

Métricas compartidas:
  1. [métrica 1]
  2. [métrica 2]
  3. [métrica 3]
'''

print(solucion)

## 7. Resumen

### ✅ Conceptos Clave

**Patrones arquitectónicos:**
- Soluciones probadas a problemas comunes
- Facilitan comunicación y reutilización
- Requieren entender contexto y trade-offs

**Arquitectura en Capas (Medallion):**
- 🥉 **Bronze:** Datos crudos, inmutables
- 🥈 **Silver:** Datos limpios, validados
- 🥇 **Gold:** Datos agregados, listos para BI

**Monolítica vs Distribuida:**
- **Monolítica:** Simple, pero difícil de escalar
- **Distribuida:** Escalable, pero más compleja
- Decisión depende de tamaño de equipo y requisitos

**Event-Driven:**
- Comunicación asíncrona mediante eventos
- Desacoplamiento y escalabilidad
- Útil para real-time analytics y CDC

**Reutilización:**
- Single Source of Truth (SSOT)
- Evitar duplicación de lógica y datos
- Data contracts para garantizar calidad

### 🎯 Próximos Pasos (Nivel Mid)

- Implementación práctica de patrones
- Bus de datos y microservicios
- Data as a Product
- Patrones de interoperabilidad
- Casos de uso empresariales

### 📚 Recursos Recomendados

- "Fundamentals of Data Engineering" - Joe Reis & Matt Housley
- "Designing Data-Intensive Applications" - Martin Kleppmann
- Databricks Medallion Architecture Guide
- Martin Fowler's blog on Event-Driven Architecture